In [1]:
# Import libraries and StackSats classes needed for exporting strategy weights, merging data, and plotting results.
import sys
import polars as pl
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
import pandas as pd

from stacksats.runner.core import StrategyRunner
from stacksats.strategy_types import ExportConfig
from stacksats.strategies.stable.baselines.uniform import UniformStrategy
from stacksats.strategies.stable.signals.momentum import MomentumStrategy

_root = Path.cwd()
while not (_root / "src").exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src import config, data_utils, plots, strategy_utils

STACKSATS_DATA_PATH = config.STACKSATS_DATA_PATH
RAW_PATH = config.RAW_PATH

data_utils.check_stacksats_data(STACKSATS_DATA_PATH, RAW_PATH)

True

In [2]:
# Initialize the runner and load the prepared Bitcoin analytics parquet manually.
runner = StrategyRunner()

btc_df = (
    pl.read_parquet(STACKSATS_DATA_PATH)
    .with_columns(pl.col("date").cast(pl.Datetime))
    .sort("date")
)

print("Loaded rows:", btc_df.height)

print(
    btc_df.select(
        pl.col("date").min().alias("min_date"),
        pl.col("date").max().alias("max_date")
    )
)

Loaded rows: 5689
shape: (1, 2)
┌─────────────────────┬─────────────────────┐
│ min_date            ┆ max_date            │
│ ---                 ┆ ---                 │
│ datetime[μs]        ┆ datetime[μs]        │
╞═════════════════════╪═════════════════════╡
│ 2010-08-16 00:00:00 ┆ 2026-03-13 00:00:00 │
└─────────────────────┴─────────────────────┘


In [3]:
# Restrict the data to the requested overall horizon while respecting actual available coverage.
btc_full = (
    btc_df
    .filter(
        (pl.col("date") >= pl.datetime(2010, 8, 16)) &
        (pl.col("date") <= pl.datetime(2023, 12, 31)) &
        pl.col("price_usd").is_not_null()
    )
    .sort("date")
)

print("Filtered rows:", btc_full.height)

print(
    btc_full.select(
        pl.col("date").min().alias("min_date"),
        pl.col("date").max().alias("max_date")
    )
)

Filtered rows: 4886
shape: (1, 2)
┌─────────────────────┬─────────────────────┐
│ min_date            ┆ max_date            │
│ ---                 ┆ ---                 │
│ datetime[μs]        ┆ datetime[μs]        │
╞═════════════════════╪═════════════════════╡
│ 2010-08-16 00:00:00 ┆ 2023-12-31 00:00:00 │
└─────────────────────┴─────────────────────┘


In [4]:
momentum_strategy = MomentumStrategy()
uniform_strategy = UniformStrategy()

momentum_yearly = []
uniform_yearly = []

for year in range(2010, 2024):
    momentum_result = strategy_utils.export_one_year(momentum_strategy, btc_full, year, runner)
    if momentum_result is not None:
        momentum_yearly.append(momentum_result)

    uniform_result = strategy_utils.export_one_year(uniform_strategy, btc_full, year, runner)
    if uniform_result is not None:
        uniform_yearly.append(uniform_result)

print("Momentum valid yearly exports:", len(momentum_yearly))
print("Uniform valid yearly exports:", len(uniform_yearly))

2010: skipped, less than 365 rows
2010: skipped, less than 365 rows
2011: exported 365 rows
2011: exported 365 rows
2012: exported 365 rows
2012: exported 365 rows
2013: exported 365 rows
2013: exported 365 rows
2014: exported 365 rows
2014: exported 365 rows
2015: exported 365 rows
2015: exported 365 rows
2016: exported 365 rows
2016: exported 365 rows
2017: exported 365 rows
2017: exported 365 rows
2018: exported 365 rows
2018: exported 365 rows
2019: exported 365 rows
2019: exported 365 rows
2020: exported 365 rows
2020: exported 365 rows
2021: exported 365 rows
2021: exported 365 rows
2022: exported 365 rows
2022: exported 365 rows
2023: exported 365 rows
2023: exported 365 rows
Momentum valid yearly exports: 13
Uniform valid yearly exports: 13


In [5]:
if not momentum_yearly:
    raise ValueError("No valid Momentum exports were produced.")

if not uniform_yearly:
    raise ValueError("No valid Uniform exports were produced.")

momentum_all = pl.concat(momentum_yearly).rename({"weight": "momentum_weight_raw"})
uniform_all = pl.concat(uniform_yearly).rename({"weight": "baseline_weight_raw"})

print("Momentum combined rows:", momentum_all.height)
print("Uniform combined rows:", uniform_all.height)

Momentum combined rows: 4745
Uniform combined rows: 4745


In [6]:
merged = (
    momentum_all
    .select(["date", "price_usd", "momentum_weight_raw"])
    .join(
        uniform_all.select(["date", "baseline_weight_raw"]),
        on="date",
        how="inner"
    )
    .sort("date")
)

print("Merged rows:", merged.height)

merged.head()

Merged rows: 4745


date,price_usd,momentum_weight_raw,baseline_weight_raw
datetime[μs],f64,f64,f64
2011-01-01 00:00:00,0.3,0.00274,0.00274
2011-01-02 00:00:00,0.3,0.00274,0.00274
2011-01-03 00:00:00,0.3,0.00274,0.00274
2011-01-04 00:00:00,0.29,0.00274,0.00274
2011-01-05 00:00:00,0.3,0.00274,0.00274


In [7]:
momentum_sum = merged["momentum_weight_raw"].sum()
baseline_sum = merged["baseline_weight_raw"].sum()

merged = merged.with_columns([
    (pl.col("momentum_weight_raw") / momentum_sum).alias("momentum_weight"),
    (pl.col("baseline_weight_raw") / baseline_sum).alias("baseline_weight"),
])

print("Momentum normalized weight sum:", merged["momentum_weight"].sum())
print("Baseline normalized weight sum:", merged["baseline_weight"].sum())

Momentum normalized weight sum: 1.0
Baseline normalized weight sum: 1.0000000000000007


In [8]:
total_budget_usd = 1000.0

merged = merged.with_columns([
    (pl.col("momentum_weight") * total_budget_usd).alias("dynamic_usd"),
    (pl.col("baseline_weight") * total_budget_usd).alias("baseline_usd"),
])

merged = merged.with_columns([
    (pl.col("dynamic_usd") / pl.col("price_usd")).alias("btc_accum_dynamic"),
    (pl.col("baseline_usd") / pl.col("price_usd")).alias("btc_accum_baseline"),
])

merged = merged.with_columns([
    (pl.col("btc_accum_dynamic") * 100_000_000).alias("sats_accum_dynamic"),
    (pl.col("btc_accum_baseline") * 100_000_000).alias("sats_accum_baseline"),
])

merged = merged.with_columns([
    (pl.col("sats_accum_dynamic") / pl.col("dynamic_usd")).alias("sats_per_dollar_dynamic"),
    (pl.col("sats_accum_baseline") / pl.col("baseline_usd")).alias("sats_per_dollar_baseline"),
])

total_dynamic_btc = merged["btc_accum_dynamic"].sum()
total_baseline_btc = merged["btc_accum_baseline"].sum()

sats_per_dollar_dynamic = (total_dynamic_btc / total_budget_usd) * 100_000_000
sats_per_dollar_baseline = (total_baseline_btc / total_budget_usd) * 100_000_000

pct_diff_vs_baseline = (
    (total_dynamic_btc - total_baseline_btc) / total_baseline_btc
) * 100

performance_label = "better" if pct_diff_vs_baseline > 0 else "worse"

print(f"Total BTC accumulated, Momentum: {total_dynamic_btc:.6f}")
print(f"Total BTC accumulated, DCA: {total_baseline_btc:.6f}")
print(f"Sats per dollar, Momentum: {sats_per_dollar_dynamic:.2f}")
print(f"Sats per dollar, DCA: {sats_per_dollar_baseline:.2f}")
print(f"Momentum performed {abs(pct_diff_vs_baseline):.2f}% {performance_label} than DCA")

Total BTC accumulated, Momentum: 59.483452
Total BTC accumulated, DCA: 61.399716
Sats per dollar, Momentum: 5948345.18
Sats per dollar, DCA: 6139971.61
Momentum performed 3.12% worse than DCA


In [9]:
plot_df = merged.to_pandas()
plot_df["date"] = pd.to_datetime(plot_df["date"])
plot_df["year"] = plot_df["date"].dt.year
plot_df["cycle_label"] = plot_df["date"].apply(plots.assign_cycle_label)

plot_df.head()

,date,price_usd,momentum_weight_raw,baseline_weight_raw,momentum_weight,baseline_weight,dynamic_usd,baseline_usd,btc_accum_dynamic,btc_accum_baseline,sats_accum_dynamic,sats_accum_baseline,sats_per_dollar_dynamic,sats_per_dollar_baseline,year,cycle_label
0,2011-01-01,0.30,0.00274,0.00274,0.000211,0.000211,0.210748,0.210748,0.702494,0.702494,7.024939e+07,7.024939e+07,3.333333e+08,3.333333e+08,2011,Cycle 1: 2010-2013
1,2011-01-02,0.30,0.00274,0.00274,0.000211,0.000211,0.210748,0.210748,0.702494,0.702494,7.024939e+07,7.024939e+07,3.333333e+08,3.333333e+08,2011,Cycle 1: 2010-2013
2,2011-01-03,0.30,0.00274,0.00274,0.000211,0.000211,0.210748,0.210748,0.702494,0.702494,7.024939e+07,7.024939e+07,3.333333e+08,3.333333e+08,2011,Cycle 1: 2010-2013
3,2011-01-04,0.29,0.00274,0.00274,0.000211,0.000211,0.210748,0.210748,0.726718,0.726718,7.267178e+07,7.267178e+07,3.448276e+08,3.448276e+08,2011,Cycle 1: 2010-2013
4,2011-01-05,0.30,0.00274,0.00274,0.000211,0.000211,0.210748,0.210748,0.702494,0.702494,7.024939e+07,7.024939e+07,3.333333e+08,3.333333e+08,2011,Cycle 1: 2010-2013


In [10]:
top_buy_points_list = []

for year, year_df in plot_df.groupby("year"):
    threshold = year_df["momentum_weight"].quantile(0.90)

    top_year_df = year_df[year_df["momentum_weight"] >= threshold].copy()
    top_year_df["top_buy_threshold_year"] = threshold

    top_buy_points_list.append(top_year_df)

top_buy_points = pd.concat(top_buy_points_list, ignore_index=True)

top_buy_points = top_buy_points.sort_values(
    ["year", "momentum_weight"],
    ascending=[True, False]
)

print("Top buy points:", len(top_buy_points))
top_buy_points[["date", "year", "price_usd", "momentum_weight", "top_buy_threshold_year"]].head()

Top buy points: 481


,date,year,price_usd,momentum_weight,top_buy_threshold_year
4,2011-07-09,2011,14.39,0.000516,0.000378
3,2011-07-08,2011,14.35,0.000499,0.000378
5,2011-07-10,2011,15.08,0.000452,0.000378
11,2011-08-06,2011,7.82,0.000452,0.000378
12,2011-08-07,2011,7.72,0.000440,0.000378


In [11]:
cols = plots.StrategyColumns(
    weight="momentum_weight",
    spd="sats_per_dollar_dynamic",
    sats_accum="sats_accum_dynamic",
)

# Full period
full_plot = plots.plot_strategy_full_period(plot_df, cols, "Momentum", ("2010-08-16", "2023-12-31"))
plt.show()

In [12]:
top_buy_points_table = top_buy_points[
    [
        "date",
        "year",
        "cycle_label",
        "price_usd",
        "momentum_weight",
        "top_buy_threshold_year",
    ]
].copy()

top_buy_points_table["date"] = top_buy_points_table["date"].dt.strftime("%Y-%m-%d")
top_buy_points_table["price_usd"] = top_buy_points_table["price_usd"].round(2)
top_buy_points_table["momentum_weight"] = top_buy_points_table["momentum_weight"].round(8)
top_buy_points_table["top_buy_threshold_year"] = top_buy_points_table["top_buy_threshold_year"].round(8)

top_buy_points_table.head(50)

,date,year,cycle_label,price_usd,momentum_weight,top_buy_threshold_year
4,2011-07-09,2011,Cycle 1: 2010-2013,14.39,0.000516,0.000378
3,2011-07-08,2011,Cycle 1: 2010-2013,14.35,0.000499,0.000378
5,2011-07-10,2011,Cycle 1: 2010-2013,15.08,0.000452,0.000378
11,2011-08-06,2011,Cycle 1: 2010-2013,7.82,0.000452,0.000378
12,2011-08-07,2011,Cycle 1: 2010-2013,7.72,0.000440,0.000378
13,2011-08-08,2011,Cycle 1: 2010-2013,7.74,0.000438,0.000378
24,2011-09-17,2011,Cycle 1: 2010-2013,4.77,0.000432,0.000378
23,2011-09-16,2011,Cycle 1: 2010-2013,4.81,0.000432,0.000378
22,2011-09-15,2011,Cycle 1: 2010-2013,4.97,0.000431,0.000378
25,2011-09-18,2011,Cycle 1: 2010-2013,5.11,0.000426,0.000378
